# Fine-tune YOLOv8 for Fruit & Vegetable Detection

This notebook trains a custom YOLO model to detect 50+ fruits and vegetables.

## 1. Setup & Installation

In [ ]:
!pip install ultralytics roboflow opencv-python pillow

In [ ]:
import os
from ultralytics import YOLO
import yaml
import shutil
from pathlib import Path

## 2. Download Dataset from Roboflow

Using Roboflow's public fruit/vegetable datasets

In [ ]:
from roboflow import Roboflow

# Initialize Roboflow (get API key from https://roboflow.com)
rf = Roboflow(api_key="YOUR_API_KEY_HERE")

# Download fruit/vegetable dataset
project = rf.workspace("joseph-nelson").project("fruits-vegetables-detection")
dataset = project.version(2).download("yolov8", location="./datasets/fruits")

print(f"Dataset downloaded to: {dataset.location}")

## 3. Alternative: Use Custom Dataset

If you have your own images, organize them like this:
```
datasets/custom/
├── train/
│   ├── images/
│   └── labels/
├── valid/
│   ├── images/
│   └── labels/
└── data.yaml
```

In [ ]:
# Create custom dataset structure
def create_dataset_structure():
    base = Path('datasets/custom')
    for split in ['train', 'valid', 'test']:
        (base / split / 'images').mkdir(parents=True, exist_ok=True)
        (base / split / 'labels').mkdir(parents=True, exist_ok=True)
    print("Dataset structure created")

# create_dataset_structure()  # Uncomment to use

## 4. Create data.yaml Configuration

In [ ]:
# Define your classes (50+ fruits/vegetables)
classes = [
    'apple', 'banana', 'orange', 'strawberry', 'grape', 'watermelon', 'pineapple',
    'mango', 'kiwi', 'peach', 'pear', 'cherry', 'plum', 'lemon', 'lime',
    'blueberry', 'raspberry', 'blackberry', 'papaya', 'coconut', 'avocado',
    'pomegranate', 'cantaloupe', 'tomato', 'cucumber', 'carrot', 'broccoli',
    'cauliflower', 'lettuce', 'cabbage', 'spinach', 'kale', 'celery',
    'bell pepper', 'onion', 'garlic', 'potato', 'sweet potato', 'corn',
    'peas', 'green beans', 'eggplant', 'zucchini', 'pumpkin', 'mushroom',
    'asparagus', 'radish', 'beet', 'turnip', 'ginger'
]

# Create data.yaml
data_yaml = {
    'path': './datasets/fruits',  # dataset root
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': len(classes),
    'names': classes
}

with open('datasets/fruits/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

print(f"Created data.yaml with {len(classes)} classes")

## 5. Load Pre-trained Model

In [ ]:
# Load pre-trained YOLOv8 model (choose size based on your needs)
# n=nano, s=small, m=medium, l=large, x=extra large

model = YOLO('yolov8m.pt')  # Medium model - good balance
# model = YOLO('yolov8n.pt')  # Faster, less accurate
# model = YOLO('yolov8x.pt')  # Slower, more accurate

print("Model loaded successfully")

## 6. Train the Model

In [ ]:
# Training configuration
results = model.train(
    data='datasets/fruits/data.yaml',
    epochs=100,              # Number of training epochs
    imgsz=640,              # Image size
    batch=16,               # Batch size (adjust based on GPU memory)
    name='fruit_vegetable_detector',
    patience=20,            # Early stopping patience
    save=True,              # Save checkpoints
    device='mps',           # Use 'mps' for M1 Mac, 'cuda' for NVIDIA GPU, 'cpu' for CPU
    workers=4,              # Number of workers
    project='runs/train',   # Project directory
    exist_ok=True,
    pretrained=True,        # Use pre-trained weights
    optimizer='Adam',       # Optimizer
    lr0=0.001,             # Initial learning rate
    weight_decay=0.0005,   # Weight decay
    warmup_epochs=3,       # Warmup epochs
    cos_lr=True,           # Cosine learning rate scheduler
    augment=True,          # Data augmentation
    hsv_h=0.015,           # HSV-Hue augmentation
    hsv_s=0.7,             # HSV-Saturation augmentation
    hsv_v=0.4,             # HSV-Value augmentation
    degrees=10,            # Rotation augmentation
    translate=0.1,         # Translation augmentation
    scale=0.5,             # Scale augmentation
    fliplr=0.5,            # Horizontal flip probability
    mosaic=1.0,            # Mosaic augmentation
    mixup=0.1              # Mixup augmentation
)

print("Training completed!")

## 7. Evaluate Model Performance

In [ ]:
# Validate the model
metrics = model.val()

print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall: {metrics.box.mr:.3f}")

## 8. Test on Sample Images

In [ ]:
# Test on validation images
results = model.predict(
    source='datasets/fruits/valid/images',
    conf=0.25,
    save=True,
    project='runs/predict',
    name='test_results'
)

print(f"Predictions saved to: runs/predict/test_results")

## 9. Export Model for Production

In [ ]:
# Export to different formats
best_model = YOLO('runs/train/fruit_vegetable_detector/weights/best.pt')

# Export to PyTorch format (default)
best_model.export(format='torchscript')

# Export to ONNX (for cross-platform)
# best_model.export(format='onnx')

# Export to CoreML (for iOS/macOS)
# best_model.export(format='coreml')

print("Model exported successfully")

## 10. Copy Model to Project

In [ ]:
# Copy best model to models folder
source = 'runs/train/fruit_vegetable_detector/weights/best.pt'
destination = 'models/best.pt'

shutil.copy(source, destination)
print(f"Model copied to {destination}")
print("\nUpdate config.json to use this model:")
print('"path": "models/best.pt"')

## 11. Quick Test with Webcam

In [ ]:
import cv2

# Load trained model
trained_model = YOLO('models/best.pt')

# Test with webcam
cap = cv2.VideoCapture(0)

print("Press 'q' to quit")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    results = trained_model(frame, conf=0.25, verbose=False)
    annotated = results[0].plot()
    
    cv2.imshow('Fine-tuned Model Test', annotated)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("Test completed")

## 12. Training Tips

### For Better Results:
- **More data**: 100+ images per class minimum
- **Balanced dataset**: Similar number of images per class
- **Data augmentation**: Enabled by default
- **Longer training**: 100-200 epochs
- **Larger model**: Use yolov8l or yolov8x for better accuracy

### If Training is Slow:
- Reduce batch size
- Use smaller model (yolov8n)
- Reduce image size to 416

### If Overfitting:
- Increase dropout
- More data augmentation
- Early stopping (patience parameter)
- Reduce model size